# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. It will guide you through accessing the data, reviewing key metadata and structure, extracting records by `@id`, and performing common analyses.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object, not a dict

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We use each entity's `@id` for precise referencing. Let's inspect all record sets and their field `@id`s.

In [ ]:
# List all record sets and their fields using @id
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets defined in the metadata. Attempting to infer from data...')

all_record_set_ids = []  # Store all discovered record set @ids

# mlcroissant 1.0 often expects at least one record set, but sometimes it's called 'hasPart' for tables/CSVs
if hasattr(metadata, 'hasPart'):
    for obj in metadata.hasPart:
        if hasattr(obj, '@id'):
            all_record_set_ids.append(obj['@id'] if isinstance(obj, dict) else getattr(obj, '@id', str(obj)))
            print(f"Found (possible) record set: @id = {all_record_set_ids[-1]}")
else:
    # If the list is empty, try known convention for standard datasets: 'cr:main', etc.
    print('No hasPart field found either.')
    print('Try listing a few records to see what names are recognized...')

# If we still have nothing, empirically trying a standard id as an educated guess
if not all_record_set_ids:
    try_ids = ['cr:main', 'cr:table', 'cr:recordSet', 'cr:protein_table']
    for try_id in try_ids:
        try:
            # Try fetching at least one record
            record = next(dataset.records(record_set=try_id))
            all_record_set_ids.append(try_id)
            print(f"Guessed record set present: @id = {try_id}")
            break
        except Exception as err:
            continue
if not all_record_set_ids:
    print("Unable to automatically determine record set IDs. Please consult the dataset documentation or metadata JSON.")

# If record set exist, you may inspect the first record and print its available keys (these are field @ids)
for recset_id in all_record_set_ids:
    print(f"\nFields for record set {recset_id}:")
    # Look at one record
    recs = dataset.records(record_set=recset_id)
    try:
        first = next(recs)
        for k in first.keys():
            print(f"  - {k}")
    except Exception:
        print("  (no records or unable to display)")

## 3. Data Extraction
Load data from a discovered record set into a pandas DataFrame for analysis. All fields and columns will be referenced by their `@id`.

**Note:** If multiple record sets are present, data from each will be loaded into a DataFrame in a dictionary keyed by record set `@id`.

In [ ]:
# Extract all discovered record sets into DataFrames using `@id`
if not all_record_set_ids:
    raise RuntimeError("No data record sets found for extraction.")

dataframes = {}
for recset_id in all_record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"Loaded record set {recset_id}: {dataframes[recset_id].shape[0]} records, columns: {list(dataframes[recset_id].columns)}\n")

# For exploration, focus on the first discovered record set
main_record_set = all_record_set_ids[0]
print(f"Sample data from record set {main_record_set}:\n")
display_columns = dataframes[main_record_set].columns.tolist()
print(display_columns)
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—filtering, normalizing, categorizing—using `@id` fields for references.

Below, we select a numeric field for demonstration (e.g., abundance value, peptide count, or other numeric column—refer to the printout from the previous section for actual field IDs).

In [ ]:
# Choose a numeric field `@id` (replace with an actual column, e.g., 'cr:abundance' if present)
possible_numeric_fields = [col for col in dataframes[main_record_set].columns if dataframes[main_record_set][col].dtype in ['float64', 'int64', 'float32', 'int32']]
if len(possible_numeric_fields) == 0:
    # Fallback, try to cast typical numeric columns
    for col in dataframes[main_record_set].columns:
        try:
            dataframes[main_record_set][col] = pd.to_numeric(dataframes[main_record_set][col])
        except Exception:
            continue
    possible_numeric_fields = [col for col in dataframes[main_record_set].columns if dataframes[main_record_set][col].dtype in ['float64', 'int64', 'float32', 'int32']]

# Use the first found numeric field for analysis
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    raise ValueError('No numeric field found for EDA! Please check available columns.')

# Filter for values greater than a threshold (example: 10)
threshold = 10
filtered_df = dataframes[main_record_set][dataframes[main_record_set][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {filtered_df.shape[0]} remaining")
print(filtered_df.head())

# Normalize the chosen field
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Try grouping by a categorical field if present. We'll guess one by looking for columns with string/object dtype.
possible_group_fields = [col for col in dataframes[main_record_set].columns if dataframes[main_record_set][col].dtype == 'O' and col != numeric_field_id]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=numeric_field_id+'_mean')
    print(f"Grouped data by {group_field} (showing mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print('No suitable group field found for grouping analysis.')

## 5. Visualization
Visualize numeric field distribution and/or grouped means. You can adjust fields for your dataset as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (before filtering)
plt.figure(figsize=(6, 4))
sns.histplot(dataframes[main_record_set][numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If we did a grouped mean, show a bar plot of means (if less than 20 groups)
if group_field and 'grouped_df' in locals() and grouped_df.shape[0] <= 20:
    grouped_df.reset_index().plot(
        kind='bar',
        x=group_field,
        y=numeric_field_id+'_mean',
        legend=False,
        figsize=(7, 4),
        title=f"Mean {numeric_field_id} by {group_field}"
    )
    plt.ylabel(numeric_field_id + ' (mean)')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We've demonstrated the process of accessing, inspecting, and analyzing your dataset using `mlcroissant`, with field and record set access by `@id`. You can extend this notebook for advanced EDA, machine learning, or domain-specific analyses!